In [ ]:
import os, sys
# import utils.data_processing as dp
# import utils.pair_methods as pm
import scipy
from datetime import datetime
import re
from collections import defaultdict, OrderedDict
#torch
import torch
from torchvision.models import alexnet
from torchvision import transforms as T
from PIL import Image
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
from sklearn.metrics.pairwise import cosine_similarity
from scipy.optimize import linear_sum_assignment
import scipy.optimize as optimize

# from dwave.samplers import SimulatedAnnealingSampler

# import dimod
from scipy.linalg import block_diag
from scipy.spatial.distance import cdist
from munkres import Munkres, print_matrix
# Qiskit
import qiskit_optimization as qo
from qiskit_optimization import QuadraticProgram

# quimb



# Problem modelling imports
from docplex.mp.model import Model

# Qiskit imports
from qiskit_algorithms import QAOA, NumPyMinimumEigensolver
from qiskit_algorithms.optimizers import COBYLA
from qiskit_algorithms.utils import algorithm_globals
from qiskit.primitives import Sampler
from qiskit_optimization.algorithms import MinimumEigenOptimizer, CplexOptimizer
from qiskit_optimization import QuadraticProgram
from qiskit_optimization.problems.variable import VarType
from qiskit_optimization.converters.quadratic_program_to_qubo import QuadraticProgramToQubo
from qiskit_optimization.translators import from_docplex_mp


In [2]:
#search for and find the files
def path_joiner(image_name, root_dir = None): #recursive search for the image
    if root_dir is None:
        root_dir = os.getcwd()
    for dirpath , _, filenames in os.walk(root_dir):
        if image_name in filenames:
            return os.path.join(dirpath, image_name)
#usage
mat_files = ['Cars_006a.mat','Cars_007a.mat','Cars_008b.mat']
paths = []
for fi in mat_files:
    paths.append(path_joiner(fi))

In [3]:
#Keypoints extraction
def keypoint(image_path, max_points=None): # extract a desired number of keypoints from the images
    keypoints1 = scipy.io.loadmat(image_path)["pts_coord"]
    if max_points is not None:
        keypoints1 = keypoints1[:, :max_points]
    return keypoints1
def keypoints_dict(image_names: list, max_points: int):
    keypoints = {}
    for image_name in image_names:
        base_name = os.path.splitext(image_name)[0]
        full_path = path_joiner(image_name)
        if full_path:
            kps = keypoint(full_path, max_points)
            keypoints[base_name] = kps
        else:
            print(f"[Warning] image not found: {image_name}")
    return keypoints
points_dic = keypoints_dict(mat_files,max_points=3)
print(f'This is the keypoints dict {points_dic}')

This is the keypoints dict {'Cars_006a': array([[ 47.93528064, 151.86426117,  45.965063  ],
       [ 98.15864834, 121.80126002,  57.2766323 ]]), 'Cars_007a': array([[ 38.44802   , 128.33906116,  24.28337108],
       [159.21073186, 177.73373428, 115.62719675]]), 'Cars_008b': array([[ 24.22058824, 117.125     ,  14.44117647],
       [167.35845588, 173.22610294, 143.39889706]])}


In [4]:
#Feature extraction using AlexNet

def alexnet_feature_extractor(layer= 'conv4'):
    model = alexnet(pretrained=True)
    model.eval()
    if layer == 'conv4':
        return torch.nn.Sequential(*list(model.features)[:10])
    elif layer == 'conv5':
        return torch.nn.Sequential(*list(model.features)[:12])
    else:
        raise ValueError("Invalid layer. Choose 'conv4' or 'conv5'.")
def extract_features(keypoints_dict, patch_size=64, layer='conv4'):
    feature_extractor = alexnet_feature_extractor(layer)
    transform = T.Compose([
        T.Resize((227, 227)),
        T.ToTensor(),
        T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    feature_extractor.to(device)
    features = {}

    for image_name, keypoints in keypoints_dict.items():
        img_path = path_joiner(image_name + '.png')
        img = Image.open(img_path).convert('RGB')
        img_np = np.array(img)
        feature_list = []

        for (x, y) in keypoints.T:
            x, y = int(round(x)), int(round(y))
            x1 = max(0, x - patch_size // 2)
            y1 = max(0, y - patch_size // 2)
            x2 = min(img.width, x + patch_size // 2)
            y2 = min(img.height, y + patch_size // 2)

            patch = img.crop((x1, y1, x2, y2))
            patch_tensor = transform(patch).unsqueeze(0).to(device)
            with torch.no_grad():
                feat = feature_extractor(patch_tensor)
                feat = feat.mean(dim=[2, 3]) # to flatten the output dimensions
            feature_list.append(feat.squeeze().cpu().numpy())

        features[image_name] = np.stack(feature_list)
    return features
features = extract_features(points_dic)
print(features)

c:\Users\sasan\AppData\Local\Programs\Python\Python313\Lib\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
c:\Users\sasan\AppData\Local\Programs\Python\Python313\Lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=AlexNet_Weights.IMAGENET1K_V1`. You can also use `weights=AlexNet_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


{'Cars_006a': array([[6.32796109e-01, 6.57072425e-01, 4.35006380e-01, 7.18967095e-02,
        3.05773377e-01, 2.39096507e-01, 3.63744289e-01, 9.46795270e-02,
        4.98862177e-01, 8.09586465e-01, 3.26265991e-01, 3.28796268e-01,
        8.66393328e-01, 4.46965098e-01, 1.82176188e-01, 2.13933051e-01,
        1.37901708e-01, 1.91459209e-01, 4.84022647e-01, 5.87426543e-01,
        3.54669392e-01, 2.69097537e-01, 6.08666599e-01, 1.71902955e-01,
        1.19614325e-01, 1.43517479e-01, 6.21197373e-02, 5.93448699e-01,
        4.48882848e-01, 6.71965003e-01, 5.33294976e-01, 2.66930517e-02,
        6.72206104e-01, 3.93196285e-01, 1.54066002e-02, 3.76293898e-01,
        4.14224297e-01, 1.10636264e-01, 4.08919573e-01, 3.06918770e-01,
        7.73870647e-01, 2.88729459e-01, 6.49675906e-01, 8.21976006e-01,
        2.40379184e-01, 3.41434121e-01, 8.56706440e-01, 1.52100638e-01,
        1.22340441e+00, 4.11651254e-01, 5.45349956e-01, 2.86167085e-01,
        9.83565152e-01, 3.40445668e-01, 1.39378950

In [ ]:
#features have been extracted, now we need to calculate the distance matrix
def pairwise_permutations(features_dict, pm_size: int) -> dict: # compute the P_ij
    P = {}
    image_list = list(features_dict.keys())

    for i in range(len(image_list)):
        key0= f"P{i+1}{i+1}"
        P[key0] = np.eye(pm_size)
        for j in range(i + 1, len(image_list)):
            img1 = image_list[i]
            img2 = image_list[j]

            feats1 = features_dict[img1]
            feats2 = features_dict[img2]
            similarity = cosine_similarity(feats1, feats2)
            cost_mat = (1 - similarity)
            row_ind, col_ind = linear_sum_assignment(cost_mat)
            n = len(row_ind)
            perm_matrix = np.zeros((n, n), dtype=int)
            perm_matrix[row_ind, col_ind] = 1
            key1 = f"P{i+1}{j+1}"
            P[key1] = perm_matrix
            key2 = f"P{j+1}{i+1}"
            P[key2] = perm_matrix.T
    return P
P = pairwise_permutations(features, 3)
print(len(P))
print(P)

9
{'P11': array([[1., 0., 0.],
       [0., 1., 0.],
       [0., 0., 1.]]), 'P12': array([[1, 0, 0],
       [0, 1, 0],
       [0, 0, 1]]), 'P21': array([[1, 0, 0],
       [0, 1, 0],
       [0, 0, 1]]), 'P13': array([[1, 0, 0],
       [0, 1, 0],
       [0, 0, 1]]), 'P31': array([[1, 0, 0],
       [0, 1, 0],
       [0, 0, 1]]), 'P22': array([[1., 0., 0.],
       [0., 1., 0.],
       [0., 0., 1.]]), 'P23': array([[1, 0, 0],
       [0, 0, 1],
       [0, 1, 0]]), 'P32': array([[1, 0, 0],
       [0, 0, 1],
       [0, 1, 0]]), 'P33': array([[1., 0., 0.],
       [0., 1., 0.],
       [0., 0., 1.]])}


In [6]:
#Qubo formulation using Qiskit-optimization
def qubo_form_maker(P: dict, num_views: int, num_keypoints: int, penalty: float=1.5):
    # without setting the X1 to identity
    mdl = QuadraticProgram("QPS")
    m = num_views
    n = num_keypoints
    N = n * n
    #size of the x vector num_viewsx(num_keypoints^2)
    for i in range(m):
        for j in range(N):
            mdl.binary_var(name=f'x{i}{j}')
    
    #constructing the Q_prime (m.N)x(m.N)
    Q_prime = np.zeros((m*N, m*N))
    for key, p_mat_ij in P.items():
        # extract i and j from key like "P12"
        i = int(key[1]) - 1  # subtract 1 for zero-based indexing
        j = int(key[2]) - 1
        block_ij = np.kron(np.eye(n), p_mat_ij)
        for u in range(N):
            for v in range(N):
                Q_prime[i*N + u, j*N + v] -= block_ij[u, v]
    # constructing the A and b as mentioned in the paper
    A = []
    b = []

    for i in range(m):
        #Row constraints
        for row in range(n):
            constraint = np.zeros(m*N)
            for col in range(n):
                constraint[i*N+row*n+col] = 1
                A.append(constraint)
                b.append(1)
        # Column constraint
        for col in range(n):
            constraint = np.zeros(m*N)
            for row in range(n):
                idx = i * N + row * n + col
                constraint[idx] = 1
            A.append(constraint)
            b.append(1)
    A = np.array(A)
    b = np.array(b)

    #constructing Q and s
    Q = Q_prime + penalty * A.T @ A
    s = -2 * penalty * A.T @ b
    # putting things together
    
    
    mdl.minimize(linear=s, quadratic=Q)
    return mdl


#Run
qubo_model = qubo_form_maker(P, 3, 3, 1.5)
print(qubo_model.prettyprint())
print(type(qubo_model))

Problem name: QPS

Minimize
  5*x00^2 + 9*x00*x01 + 9*x00*x02 + 3*x00*x03 + 3*x00*x06 - 2*x00*x10
  - 2*x00*x20 + 5*x01^2 + 9*x01*x02 + 3*x01*x04 + 3*x01*x07 - 2*x01*x11
  - 2*x01*x21 + 5*x02^2 + 3*x02*x05 + 3*x02*x08 - 2*x02*x12 - 2*x02*x22
  + 5*x03^2 + 9*x03*x04 + 9*x03*x05 + 3*x03*x06 - 2*x03*x13 - 2*x03*x23
  + 5*x04^2 + 9*x04*x05 + 3*x04*x07 - 2*x04*x14 - 2*x04*x24 + 5*x05^2
  + 3*x05*x08 - 2*x05*x15 - 2*x05*x25 + 5*x06^2 + 9*x06*x07 + 9*x06*x08
  - 2*x06*x16 - 2*x06*x26 + 5*x07^2 + 9*x07*x08 - 2*x07*x17 - 2*x07*x27
  + 5*x08^2 - 2*x08*x18 - 2*x08*x28 + 5*x10^2 + 9*x10*x11 + 9*x10*x12
  + 3*x10*x13 + 3*x10*x16 - 2*x10*x20 + 5*x11^2 + 9*x11*x12 + 3*x11*x14
  + 3*x11*x17 - 2*x11*x22 + 5*x12^2 + 3*x12*x15 + 3*x12*x18 - 2*x12*x21
  + 5*x13^2 + 9*x13*x14 + 9*x13*x15 + 3*x13*x16 - 2*x13*x23 + 5*x14^2
  + 9*x14*x15 + 3*x14*x17 - 2*x14*x25 + 5*x15^2 + 3*x15*x18 - 2*x15*x24
  + 5*x16^2 + 9*x16*x17 + 9*x16*x18 - 2*x16*x26 + 5*x17^2 + 9*x17*x18
  - 2*x17*x28 + 5*x18^2 - 2*x18*x27 + 5*x20^2 

In [7]:
import copy
# Using warm starting 
def relax_problem(problem) -> QuadraticProgram:
    """Change all variables to continuous."""
    relaxed_problem = copy.deepcopy(problem)
    for variable in relaxed_problem.variables:
        variable.vartype = VarType.CONTINUOUS

    return relaxed_problem

In [8]:
# run
qubo_model_relaxed = relax_problem(qubo_model)
print(qubo_model_relaxed.prettyprint())

Problem name: QPS

Minimize
  5*x00^2 + 9*x00*x01 + 9*x00*x02 + 3*x00*x03 + 3*x00*x06 - 2*x00*x10
  - 2*x00*x20 + 5*x01^2 + 9*x01*x02 + 3*x01*x04 + 3*x01*x07 - 2*x01*x11
  - 2*x01*x21 + 5*x02^2 + 3*x02*x05 + 3*x02*x08 - 2*x02*x12 - 2*x02*x22
  + 5*x03^2 + 9*x03*x04 + 9*x03*x05 + 3*x03*x06 - 2*x03*x13 - 2*x03*x23
  + 5*x04^2 + 9*x04*x05 + 3*x04*x07 - 2*x04*x14 - 2*x04*x24 + 5*x05^2
  + 3*x05*x08 - 2*x05*x15 - 2*x05*x25 + 5*x06^2 + 9*x06*x07 + 9*x06*x08
  - 2*x06*x16 - 2*x06*x26 + 5*x07^2 + 9*x07*x08 - 2*x07*x17 - 2*x07*x27
  + 5*x08^2 - 2*x08*x18 - 2*x08*x28 + 5*x10^2 + 9*x10*x11 + 9*x10*x12
  + 3*x10*x13 + 3*x10*x16 - 2*x10*x20 + 5*x11^2 + 9*x11*x12 + 3*x11*x14
  + 3*x11*x17 - 2*x11*x22 + 5*x12^2 + 3*x12*x15 + 3*x12*x18 - 2*x12*x21
  + 5*x13^2 + 9*x13*x14 + 9*x13*x15 + 3*x13*x16 - 2*x13*x23 + 5*x14^2
  + 9*x14*x15 + 3*x14*x17 - 2*x14*x25 + 5*x15^2 + 3*x15*x18 - 2*x15*x24
  + 5*x16^2 + 9*x16*x17 + 9*x16*x18 - 2*x16*x26 + 5*x17^2 + 9*x17*x18
  - 2*x17*x28 + 5*x18^2 - 2*x18*x27 + 5*x20^2 

In [9]:
from qiskit_optimization.algorithms import WarmStartQAOAOptimizer, SlsqpOptimizer
from qiskit.primitives import Estimator



In [10]:
# 1) Create any continuous presolver

slsqp = SlsqpOptimizer()          # Pure SciPy
relaxed_solution = slsqp.solve(qubo_model_relaxed)
c_stars = relaxed_solution.x      # numpy array in [0,1]
print(c_stars)


[0.4 0.4 0.4 0.4 0.4 0.4 0.4 0.4 0.4 0.4 0.4 0.4 0.4 0.4 0.4 0.4 0.4 0.4
 0.4 0.4 0.4 0.4 0.4 0.4 0.4 0.4 0.4]


In [11]:
algorithm_globals.random_seed = 12345
# qaoa_mes = QAOA(sampler=Sampler(), optimizer=COBYLA(), initial_point=[0.0, 1.0])
# exact_mes = NumPyMinimumEigensolver()
# qaoa = MinimumEigenOptimizer(qaoa_mes)
# qaoa_result = qaoa.solve(qubo_model)
# print(qaoa_result.prettyprint())

In [12]:
# creating warm_starting QAOA circuit
from qiskit import QuantumCircuit

thetas = [2 * np.arcsin(np.sqrt(c_star)) for c_star in c_stars]

init_qc = QuantumCircuit(27)
for idx, theta in enumerate(thetas):
    init_qc.ry(theta, idx)

init_qc.draw()

┌────────────┐
 q_0: ┤ Ry(1.3694) ├
      ├────────────┤
 q_1: ┤ Ry(1.3694) ├
      ├────────────┤
 q_2: ┤ Ry(1.3694) ├
      ├────────────┤
 q_3: ┤ Ry(1.3694) ├
      ├────────────┤
 q_4: ┤ Ry(1.3694) ├
      ├────────────┤
 q_5: ┤ Ry(1.3694) ├
      ├────────────┤
 q_6: ┤ Ry(1.3694) ├
      ├────────────┤
 q_7: ┤ Ry(1.3694) ├
      ├────────────┤
 q_8: ┤ Ry(1.3694) ├
      ├────────────┤
 q_9: ┤ Ry(1.3694) ├
      ├────────────┤
q_10: ┤ Ry(1.3694) ├
      ├────────────┤
q_11: ┤ Ry(1.3694) ├
      ├────────────┤
q_12: ┤ Ry(1.3694) ├
      ├────────────┤
q_13: ┤ Ry(1.3694) ├
      ├────────────┤
q_14: ┤ Ry(1.3694) ├
      ├────────────┤
q_15: ┤ Ry(1.3694) ├
      ├────────────┤
q_16: ┤ Ry(1.3694) ├
      ├────────────┤
q_17: ┤ Ry(1.3694) ├
      ├────────────┤
q_18: ┤ Ry(1.3694) ├
      ├────────────┤
q_19: ┤ Ry(1.3694) ├
      ├────────────┤
q_20: ┤ Ry(1.3694) ├
      ├────────────┤
q_21: ┤ Ry(1.3694) ├
      ├────────────┤
q_22: ┤ Ry(1.3694) ├
      ├────────────┤
q_23: ┤ Ry(1.3694) ├
      ├────────────┤
q_24: ┤ Ry(1.3694) ├
      ├────────────┤
q_25: ┤ Ry(1.3694) ├
      ├────────────┤
q_26: ┤ Ry(1.3694) ├
      └────────────┘

In [13]:
#create the mixer
from qiskit.circuit import Parameter

beta = Parameter("β")

ws_mixer = QuantumCircuit(27)
for idx, theta in enumerate(thetas):
    ws_mixer.ry(-theta, idx)
    ws_mixer.rz(-2 * beta, idx)
    ws_mixer.ry(theta, idx)

ws_mixer.draw()

┌─────────────┐┌──────────┐┌────────────┐
 q_0: ┤ Ry(-1.3694) ├┤ Rz(-2*β) ├┤ Ry(1.3694) ├
      ├─────────────┤├──────────┤├────────────┤
 q_1: ┤ Ry(-1.3694) ├┤ Rz(-2*β) ├┤ Ry(1.3694) ├
      ├─────────────┤├──────────┤├────────────┤
 q_2: ┤ Ry(-1.3694) ├┤ Rz(-2*β) ├┤ Ry(1.3694) ├
      ├─────────────┤├──────────┤├────────────┤
 q_3: ┤ Ry(-1.3694) ├┤ Rz(-2*β) ├┤ Ry(1.3694) ├
      ├─────────────┤├──────────┤├────────────┤
 q_4: ┤ Ry(-1.3694) ├┤ Rz(-2*β) ├┤ Ry(1.3694) ├
      ├─────────────┤├──────────┤├────────────┤
 q_5: ┤ Ry(-1.3694) ├┤ Rz(-2*β) ├┤ Ry(1.3694) ├
      ├─────────────┤├──────────┤├────────────┤
 q_6: ┤ Ry(-1.3694) ├┤ Rz(-2*β) ├┤ Ry(1.3694) ├
      ├─────────────┤├──────────┤├────────────┤
 q_7: ┤ Ry(-1.3694) ├┤ Rz(-2*β) ├┤ Ry(1.3694) ├
      ├─────────────┤├──────────┤├────────────┤
 q_8: ┤ Ry(-1.3694) ├┤ Rz(-2*β) ├┤ Ry(1.3694) ├
      ├─────────────┤├──────────┤├────────────┤
 q_9: ┤ Ry(-1.3694) ├┤ Rz(-2*β) ├┤ Ry(1.3694) ├
      ├─────────────┤├──────────┤├────────────┤
q_10: ┤ Ry(-1.3694) ├┤ Rz(-2*β) ├┤ Ry(1.3694) ├
      ├─────────────┤├──────────┤├────────────┤
q_11: ┤ Ry(-1.3694) ├┤ Rz(-2*β) ├┤ Ry(1.3694) ├
      ├─────────────┤├──────────┤├────────────┤
q_12: ┤ Ry(-1.3694) ├┤ Rz(-2*β) ├┤ Ry(1.3694) ├
      ├─────────────┤├──────────┤├────────────┤
q_13: ┤ Ry(-1.3694) ├┤ Rz(-2*β) ├┤ Ry(1.3694) ├
      ├─────────────┤├──────────┤├────────────┤
q_14: ┤ Ry(-1.3694) ├┤ Rz(-2*β) ├┤ Ry(1.3694) ├
      ├─────────────┤├──────────┤├────────────┤
q_15: ┤ Ry(-1.3694) ├┤ Rz(-2*β) ├┤ Ry(1.3694) ├
      ├─────────────┤├──────────┤├────────────┤
q_16: ┤ Ry(-1.3694) ├┤ Rz(-2*β) ├┤ Ry(1.3694) ├
      ├─────────────┤├──────────┤├────────────┤
q_17: ┤ Ry(-1.3694) ├┤ Rz(-2*β) ├┤ Ry(1.3694) ├
      ├─────────────┤├──────────┤├────────────┤
q_18: ┤ Ry(-1.3694) ├┤ Rz(-2*β) ├┤ Ry(1.3694) ├
      ├─────────────┤├──────────┤├────────────┤
q_19: ┤ Ry(-1.3694) ├┤ Rz(-2*β) ├┤ Ry(1.3694) ├
      ├─────────────┤├──────────┤├────────────┤
q_20: ┤ Ry(-1.3694) ├┤ Rz(-2*β) ├┤ Ry(1.3694) ├
      ├─────────────┤├──────────┤├────────────┤
q_21: ┤ Ry(-1.3694) ├┤ Rz(-2*β) ├┤ Ry(1.3694) ├
      ├─────────────┤├──────────┤├────────────┤
q_22: ┤ Ry(-1.3694) ├┤ Rz(-2*β) ├┤ Ry(1.3694) ├
      ├─────────────┤├──────────┤├────────────┤
q_23: ┤ Ry(-1.3694) ├┤ Rz(-2*β) ├┤ Ry(1.3694) ├
      ├─────────────┤├──────────┤├────────────┤
q_24: ┤ Ry(-1.3694) ├┤ Rz(-2*β) ├┤ Ry(1.3694) ├
      ├─────────────┤├──────────┤├────────────┤
q_25: ┤ Ry(-1.3694) ├┤ Rz(-2*β) ├┤ Ry(1.3694) ├
      ├─────────────┤├──────────┤├────────────┤
q_26: ┤ Ry(-1.3694) ├┤ Rz(-2*β) ├┤ Ry(1.3694) ├
      └─────────────┘└──────────┘└────────────┘

In [14]:
ws_qaoa_mes = QAOA(
    sampler=Sampler(),
    optimizer=COBYLA(),
    initial_state=init_qc,
    mixer=ws_mixer,
    initial_point=[0.0, 1.0],
)
ws_qaoa = MinimumEigenOptimizer(ws_qaoa_mes)

C:\Users\sasan\AppData\Local\Temp\ipykernel_14808\3478223699.py:2: DeprecationWarning: The class ``qiskit.primitives.sampler.Sampler`` is deprecated as of qiskit 1.2. It will be removed no earlier than 3 months after the release date. All implementations of the `BaseSamplerV1` interface have been deprecated in favor of their V2 counterparts. The V2 alternative for the `Sampler` class is `StatevectorSampler`.
  sampler=Sampler(),


In [15]:
ws_qaoa_result = ws_qaoa.solve(qubo_model)
print(ws_qaoa_result.prettyprint())

MemoryError: Unable to allocate 284. GiB for an array with shape (27, 134217728) and data type <U21

In [ ]:
def qubo_to_ising(model : qo.problems.quadratic_program.QuadraticProgram):
    return qo.translators.to_ising(model)

ising_model, offset = qubo_to_ising(qubo_model_relaxed)
print(ising_model)
print(offset)

SparsePauliOp(['IIIIIIIIIIIIIIIIIIIIIIIIIIZ', 'IIIIIIIIIIIIIIIIIIIIIIIIIZI', 'IIIIIIIIIIIIIIIIIIIIIIIIZII', 'IIIIIIIIIIIIIIIIIIIIIIIZIII', 'IIIIIIIIIIIIIIIIIIIIIIZIIII', 'IIIIIIIIIIIIIIIIIIIIIZIIIII', 'IIIIIIIIIIIIIIIIIIIIZIIIIII', 'IIIIIIIIIIIIIIIIIIIZIIIIIII', 'IIIIIIIIIIIIIIIIIIZIIIIIIII', 'IIIIIIIIIIIIIIIIIZIIIIIIIII', 'IIIIIIIIIIIIIIIIZIIIIIIIIII', 'IIIIIIIIIIIIIIIZIIIIIIIIIII', 'IIIIIIIIIIIIIIZIIIIIIIIIIII', 'IIIIIIIIIIIIIZIIIIIIIIIIIII', 'IIIIIIIIIIIIZIIIIIIIIIIIIII', 'IIIIIIIIIIIZIIIIIIIIIIIIIII', 'IIIIIIIIIIZIIIIIIIIIIIIIIII', 'IIIIIIIIIZIIIIIIIIIIIIIIIII', 'IIIIIIIIZIIIIIIIIIIIIIIIIII', 'IIIIIIIZIIIIIIIIIIIIIIIIIII', 'IIIIIIZIIIIIIIIIIIIIIIIIIII', 'IIIIIZIIIIIIIIIIIIIIIIIIIII', 'IIIIZIIIIIIIIIIIIIIIIIIIIII', 'IIIZIIIIIIIIIIIIIIIIIIIIIII', 'IIZIIIIIIIIIIIIIIIIIIIIIIII', 'IZIIIIIIIIIIIIIIIIIIIIIIIII', 'ZIIIIIIIIIIIIIIIIIIIIIIIIII', 'IIIIIIIIIIIIIIIIIIIIIIIIIZZ', 'IIIIIIIIIIIIIIIIIIIIIIIIZIZ', 'IIIIIIIIIIIIIIIIIIIIIIIZIIZ', 'IIIIIIIIIIIIIIIIIIIIZIIIIIZ', 'IIIIIIIIIIIIIIIIIZIIIII

In [11]:
print(type(ising_model))

<class 'qiskit.quantum_info.operators.symplectic.sparse_pauli_op.SparsePauliOp'>


ImportError: cannot import name 'BaseSampler' from 'qiskit.primitives' (d:\Programs\anaconda3\envs\quantum-sync\Lib\site-packages\qiskit\primitives\__init__.py)

In [ ]:
# from qiskit_optimization.translators import to_ising
# from qiskit.quantum_info import SparsePauliOp

# def paulisumop_to_quimb_terms(pauli_op: SparsePauliOp):
#     """
#     Converts Qiskit's SparsePauliOp to quimb-style Ising dict:
#     keys = tuple of qubit indices, values = coefficients
#     """
#     terms = {}
#     for pauli, coeff in zip(pauli_op.paulis, pauli_op.coeffs):
#         z_indices = [i for i, p in enumerate(pauli.to_label()) if p == 'Z']
#         if len(z_indices) > 0:
#             terms[tuple(z_indices)] = coeff.real
#     return terms